# Generate Token Graphs Across Dynamical Complexity

This Generate notebook creates the processed Llama-3.2-3B selected-layer attention-graph and hidden-state archives consumed by `Visualize-token-graphs-across-dynamical-complexity.ipynb`. The companion Visualize notebook generates the final-layer visualization discussed under **Attention-Induced Token-Graph Organization Across Dynamical Complexity and Context Length** in the Results, together with the early- and middle-layer views documented under **Attention-Induced Token-Graph Visualizations at Additional Transformer Layers** in the appendix.

Archives cover all ten input families at $N = 500$ and store attention-derived graph weights, Fruchterman–Reingold layout positions, and leading uncentered hidden-state SVD coordinates for layers 1, 12, and 28.


Set imports, constants, output paths, selected layers, and random seeds.


In [ ]:
import os
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import gc
import random
import re
from pathlib import Path

import networkx as nx
import numpy as np
import torch
from scipy.integrate import solve_ivp
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

# Model, analysis context, and cache paths
MODEL_NAME = 'meta-llama/Llama-3.2-3B'
REFERENCE_CONTEXT_TOKENS = 1000
ANALYSIS_CONTEXT_TOKENS = 500
CONTEXT_FRAC = ANALYSIS_CONTEXT_TOKENS / REFERENCE_CONTEXT_TOKENS  # Relative to the 1,000-token reference length
CONTEXT_DISPLAY_LABEL = f'{ANALYSIS_CONTEXT_TOKENS} tokens'
PROCESSED_DATA_DIR = Path('processed_graph_data')
CHECKPOINT_EVERY_N_LAYERS = 1
GRAPH_CHECKPOINT_SCHEMA_VERSION = 'selected_layer_attention_graph_svd_checkpoint_v1'

# Save selected attention layers
LAYERS_TO_SAVE = (1, 12, 28)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


In [ ]:

# Input families and saved-array schema
DATA_ORDER = [
    'Constant 500',
    'Logistic r=3.2',
    'Logistic r=3.5',
    'Logistic r=3.55',
    'Logistic r=3.566',
    'Logistic r=3.5692',
    'Logistic r=3.5698',
    'Logistic r=3.9',
    'Logistic r=4.0',
    'Lorenz',
]

DATA_TITLE_SUFFIXES = {
    'Logistic r=3.2': 'period 2',
    'Logistic r=3.5': 'period 4',
    'Logistic r=3.55': 'period 8',
    'Logistic r=3.566': 'period 16',
    'Logistic r=3.5692': 'period 32',
    'Logistic r=3.5698': 'period 64',
    'Logistic r=3.9': 'chaotic',
    'Logistic r=4.0': 'chaotic',
}

GRAPH_DATA_ARRAY_KEYS = (
    'layer_indices',
    'Wbar',
    'layout_positions',
    'svd_projection_coords',
    'svd_cumulative_energy_ratio',
    'is_delimiter',
)


def panel_title(data_name):
    suffix = DATA_TITLE_SUFFIXES.get(str(data_name))
    return f'{data_name} ({suffix})' if suffix else str(data_name)


def context_display_label(num_tokens=ANALYSIS_CONTEXT_TOKENS):
    return f'{int(num_tokens)} tokens'


def context_fraction_label(frac):
    pct = int(round(100 * float(frac)))
    return f'context{pct}'


def safe_data_stem(data_name, frac):
    name = str(data_name).replace('=', '_').replace('.', '_')
    name = re.sub(r'[^A-Za-z0-9]+', '_', name).strip('_').lower()
    return f'{name}_{context_fraction_label(frac)}'


def graph_data_path(data_name, frac=CONTEXT_FRAC, data_dir=PROCESSED_DATA_DIR):
    return Path(data_dir) / f'{safe_data_stem(data_name, frac)}_graph_data.npz'


def resolve_layer_indices_to_save(num_layers, layer_specs=LAYERS_TO_SAVE):
    num_layers = int(num_layers)
    resolved = []
    for spec in layer_specs:
        layer = int(spec)
        if layer < 1 or layer > num_layers:
            raise ValueError(f'Layer {layer} is outside the available one-based range 1..{num_layers}.')
        if layer not in resolved:
            resolved.append(layer)
    if not resolved:
        raise ValueError('LAYERS_TO_SAVE resolved to no layers.')
    return np.asarray(resolved, dtype=np.int32)


Build the constant, logistic-map, and Lorenz token sequences.


In [ ]:
def lorenz(t, y, sigma, rho, beta):
    return [
        sigma * (y[1] - y[0]),
        y[0] * (rho - y[2]) - y[1],
        y[0] * y[1] - beta * y[2],
    ]


sigma = 10.0
rho = 28.0
beta = 8.0 / 3.0
initial_conditions = [-8, 7, 27]
t_span = (0, 50)
t_eval = np.linspace(t_span[0], t_span[1], 500)

sol = solve_ivp(
    lorenz,
    t_span,
    initial_conditions,
    args=(sigma, rho, beta),
    t_eval=t_eval,
)

x_vals = sol.y[0]
x_min = x_vals.min()
x_max = x_vals.max()
x_integers = np.round(50 + (x_vals - x_min) / (x_max - x_min) * 900).astype(int)
N = len(x_integers)


In [ ]:
periodic_logistic_periods = {
    3.2: 2,
    3.5: 4,
    3.55: 8,
    3.566: 16,
    3.5692: 32,
    3.5698: 64,
}
logistic_rs = list(periodic_logistic_periods.keys()) + [3.9, 4.0]
logistic_expected_periods = {r: periodic_logistic_periods.get(r) for r in logistic_rs}
x0 = 0.2
logistic_long_n = 20000
logistic_settle_tol = 1e-10


def logistic_sequence(r, x0, n):
    x = np.empty(n, dtype=float)
    x[0] = x0
    for i in range(1, n):
        x[i] = r * x[i - 1] * (1 - x[i - 1])
    return x


def settle_time_float(x, p, tol=1e-10):
    x = np.asarray(x, dtype=float)
    if p is None or p <= 0 or p >= x.size:
        return None
    err = np.abs(x[p:] - x[:-p])
    suffix_max = np.maximum.accumulate(err[::-1])[::-1]
    hits = np.flatnonzero(suffix_max < tol)
    return int(hits[0]) if hits.size else None


def quantize_logistic_window(vals):
    vals = np.asarray(vals, dtype=float)
    vmin, vmax = vals.min(), vals.max()
    if vmax == vmin:
        return np.full(vals.shape[0], 500, dtype=int)
    return np.round(50 + (vals - vmin) / (vmax - vmin) * 900).astype(int)


def build_burned_in_logistic_window(r, x0, n_window, expected_period=None, n_long=20000, tol=1e-10):
    if expected_period is None:
        return logistic_sequence(r, x0, n_window), 0

    x_long = logistic_sequence(r, x0, n_long)
    t_settle = settle_time_float(x_long, expected_period, tol=tol)
    if t_settle is None:
        raise ValueError(f'Could not find settle time for r={r} with expected period {expected_period}.')
    if t_settle + n_window > x_long.size:
        raise ValueError(f'Need a longer trajectory for r={r}; settle index {t_settle} leaves fewer than {n_window} states.')
    return np.asarray(x_long[t_settle:t_settle + n_window], dtype=float), int(t_settle)


logistic_sequences = {}
logistic_integers = {}
logistic_window_start_indices = {}
for r in logistic_rs:
    vals, start_idx = build_burned_in_logistic_window(
        r,
        x0=x0,
        n_window=N,
        expected_period=logistic_expected_periods[r],
        n_long=logistic_long_n,
        tol=logistic_settle_tol,
    )
    logistic_sequences[r] = vals
    logistic_integers[r] = quantize_logistic_window(vals)
    logistic_window_start_indices[r] = int(start_idx)

input_data_types = {
    'Constant 500': np.full(N, 500, dtype=int),
    'Logistic r=3.2': logistic_integers[3.2],
    'Logistic r=3.5': logistic_integers[3.5],
    'Logistic r=3.55': logistic_integers[3.55],
    'Logistic r=3.566': logistic_integers[3.566],
    'Logistic r=3.5692': logistic_integers[3.5692],
    'Logistic r=3.5698': logistic_integers[3.5698],
    'Logistic r=3.9': logistic_integers[3.9],
    'Logistic r=4.0': logistic_integers[4.0],
    'Lorenz': x_integers,
}

assert list(input_data_types.keys()) == DATA_ORDER


Load the tokenizer, validate the separator token, and define lazy model loading.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
model_hs = None
num_layers = None
hidden_size = None

comma_token_ids = tokenizer(',', add_special_tokens=False).input_ids
if len(comma_token_ids) != 1:
    raise ValueError(f"Expected one token for ',', got {comma_token_ids}")
comma_token_id = int(comma_token_ids[0])


def load_attention_model(model_name=MODEL_NAME):
    global model_hs, num_layers, hidden_size
    if model_hs is None:
        model_hs = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map='auto',
            torch_dtype=torch.float16,
            attn_implementation='eager',
        )
        model_hs.eval()
        num_layers = int(model_hs.config.num_hidden_layers)
        hidden_size = int(model_hs.config.hidden_size)
    return model_hs


Handle BOS removal and project hidden states onto their leading SVD components.


In [ ]:

def get_token_start_index(input_ids, bos_token_id=None, drop_bos=True):
    if not drop_bos:
        return 0
    if input_ids is None or bos_token_id is None:
        return 0

    if isinstance(input_ids, torch.Tensor):
        first_id = int(input_ids.flatten()[0].item())
    else:
        first_id = int(np.asarray(input_ids).flatten()[0])

    return 1 if first_id == int(bos_token_id) else 0


def extract_hidden_signal_matrix(hidden_states, input_ids, bos_token_id=None, layer_index=-2, drop_bos=True):
    token_start = get_token_start_index(input_ids, bos_token_id=bos_token_id, drop_bos=drop_bos)

    resolved_layer_index = layer_index if layer_index >= 0 else len(hidden_states) + layer_index
    if resolved_layer_index < 0 or resolved_layer_index >= len(hidden_states):
        raise ValueError(f'Invalid layer_index={layer_index} for {len(hidden_states)} hidden-state tensors')

    signal = hidden_states[resolved_layer_index][0].float()
    signal = signal[token_start:, :]
    return signal.cpu().numpy(), int(signal.shape[0]), int(resolved_layer_index)


def svd_projection_uncentered(signal_matrix, k=3):
    X = np.asarray(signal_matrix, dtype=np.float32)
    if X.ndim != 2:
        raise ValueError('signal_matrix must be 2D (num_tokens x hidden_dim)')

    n_tokens = X.shape[0]
    coords = np.zeros((n_tokens, int(k)), dtype=np.float32)
    if n_tokens == 0:
        return {
            'coords': coords,
            'cumulative_energy_ratio': np.zeros((0,), dtype=np.float32),
        }

    U, S, _ = np.linalg.svd(X, full_matrices=False)
    S = np.asarray(S, dtype=np.float32)
    n_comp = min(int(k), S.shape[0])
    coords[:, :n_comp] = U[:, :n_comp] * S[:n_comp]

    energy = S ** 2
    total = float(np.sum(energy))
    if total <= 0:
        cumulative_energy_ratio = np.full_like(energy, np.nan, dtype=np.float32)
    else:
        cumulative_energy_ratio = (np.cumsum(energy) / total).astype(np.float32)

    return {
        'coords': coords,
        'cumulative_energy_ratio': cumulative_energy_ratio,
    }


Define context sizing, graph construction, and deterministic layout helpers.


In [ ]:
# Graph construction and layout settings
rgb_graph_layout_method = 'spring'
rgb_graph_layout_seed = 42
# Mass and uniform weighting are equivalent for the models studied here; mass also supports sparse-attention variants
rgb_graph_head_weighting = 'mass'
rgb_graph_drop_bos = True
rgb_graph_renormalize_after_drop = True
rgb_graph_layout_edge_threshold_factor = 0.0
rgb_graph_draw_edge_threshold_factor = 2.0
rgb_graph_abs_token_marker_percentiles = (0, 25, 50, 75, 100)
rgb_graph_time_init_mode = 'spiral'
rgb_graph_time_init_spring_iterations = 100
rgb_graph_time_init_pin_endpoints = False

across_layers_graph_internal_cache = {}


def _build_input_sequence_from_frac(sequence, frac, total_n):
    frac_use = float(frac)
    if frac_use >= 1.0:
        ctx_len = int(total_n)
    else:
        ctx_len = max(1, int(round(frac_use * (total_n - 1))))
    ctx_len = min(int(total_n), ctx_len)
    context_window = sequence[:ctx_len]
    input_sequence = ','.join([f'{x:03d}' for x in context_window]) + ','
    return input_sequence, ctx_len


def _get_attention_capable_model_for_graph_vis():
    return load_attention_model(MODEL_NAME)


def _edge_weight_threshold_from_uniform(n_nodes, factor):
    n_nodes = int(n_nodes)
    if n_nodes <= 0:
        return float('inf')
    return float(factor) / float(n_nodes)


def _resolve_spring_iterations(spring_iterations):
    if spring_iterations is None:
        return int(rgb_graph_time_init_spring_iterations)
    return int(spring_iterations)


def _get_abs_token_position_markers(n, percentiles=None):
    n = int(n)
    if n <= 0:
        return []

    percentiles_use = rgb_graph_abs_token_marker_percentiles if percentiles is None else tuple(percentiles)
    markers = []
    seen = set()
    for pct in percentiles_use:
        pct_float = float(pct)
        if pct_float > 1.0:
            pct_float = pct_float / 100.0
        pct_float = float(np.clip(pct_float, 0.0, 1.0))
        idx = 0 if n == 1 else int(round((n - 1) * pct_float))
        if idx in seen:
            continue
        seen.add(idx)
        markers.append((idx, int(round(100 * pct_float))))
    return markers


def _minmax_scale_rgb_channels(coords, fill_value=0.5, eps=1e-12):
    coords = np.asarray(coords, dtype=float)
    if coords.ndim == 1:
        coords = coords[:, None]
    if coords.ndim != 2:
        raise ValueError('coords must be 2D (num_tokens x num_components)')

    n = coords.shape[0]
    rgb = np.full((n, 3), float(fill_value), dtype=float)
    if n == 0:
        return rgb

    for k in range(3):
        if coords.shape[1] <= k:
            continue
        col = np.asarray(coords[:, k], dtype=float)
        finite = np.isfinite(col)
        if not np.any(finite):
            continue
        cmin = float(np.min(col[finite]))
        cmax = float(np.max(col[finite]))
        if (cmax - cmin) <= float(eps):
            rgb[finite, k] = float(fill_value)
        else:
            rgb[finite, k] = (col[finite] - cmin) / (cmax - cmin)

    rgb[~np.isfinite(rgb)] = float(fill_value)
    return np.clip(rgb, 0.0, 1.0)


def _build_time_ordered_init_positions(n, mode='spiral'):
    n = int(n)
    if n <= 0:
        return np.zeros((0, 2), dtype=float)
    if n == 1:
        return np.zeros((1, 2), dtype=float)

    t = np.linspace(0.0, 1.0, n)
    mode_use = str(mode).lower()

    if mode_use == 'circle':
        theta = -0.5 * np.pi + 2.0 * np.pi * t
        x = np.cos(theta)
        y = np.sin(theta)
    elif mode_use == 'spiral':
        turns = 2.25 if n < 80 else 2.75
        theta = -0.5 * np.pi + 2.0 * np.pi * turns * t
        r = 0.15 + 0.85 * t
        x = r * np.cos(theta)
        y = r * np.sin(theta)
    elif mode_use == 'line':
        x = -1.0 + 2.0 * t
        y = np.zeros_like(x)
    else:
        raise ValueError("mode must be one of: 'circle', 'spiral', 'line'")

    pos = np.column_stack([x, y]).astype(float)
    pos = pos - np.mean(pos, axis=0, keepdims=True)
    radius = np.max(np.linalg.norm(pos, axis=1))
    if np.isfinite(radius) and radius > 0:
        pos = pos / radius
    return pos


def _build_topological_layout_from_attention_time_init(
    Wbar,
    init_mode='spiral',
    method=None,
    seed=None,
    threshold_factor=None,
    spring_iterations=None,
    pin_endpoints=False,
):
    W = np.asarray(Wbar, dtype=float).copy()
    if W.ndim != 2 or W.shape[0] != W.shape[1]:
        raise ValueError('Wbar must be square')

    n = W.shape[0]
    if n == 0:
        return np.zeros((0, 2), dtype=float)
    if n == 1:
        return np.zeros((1, 2), dtype=float)

    W[~np.isfinite(W)] = 0.0
    W = np.clip(W, a_min=0.0, a_max=None)
    np.fill_diagonal(W, 0.0)

    factor_use = rgb_graph_layout_edge_threshold_factor if threshold_factor is None else float(threshold_factor)
    if float(factor_use) > 0.0:
        layout_threshold = _edge_weight_threshold_from_uniform(n, factor=factor_use)
        W[W <= layout_threshold] = 0.0

    init_xy = _build_time_ordered_init_positions(n, mode=init_mode)
    G = nx.from_numpy_array(W)
    if G.number_of_edges() == 0:
        return init_xy

    method_use = (rgb_graph_layout_method if method is None else method).lower()
    if method_use not in ('spring', 'fr', 'fruchterman_reingold'):
        raise ValueError("Only 'spring' layout is supported in this notebook version")

    seed_use = rgb_graph_layout_seed if seed is None else int(seed)
    pos0 = {i: init_xy[i] for i in range(n)}
    n_iter = _resolve_spring_iterations(spring_iterations)

    spring_kwargs = dict(
        weight='weight',
        seed=seed_use,
        pos=pos0,
        iterations=max(1, int(n_iter)),
    )
    if bool(pin_endpoints) and n > 1:
        spring_kwargs['fixed'] = [0, n - 1]

    pos_dict = nx.spring_layout(G, **spring_kwargs)
    pos = np.asarray([pos_dict[i] for i in range(n)], dtype=float)
    if pos.ndim == 1:
        pos = pos[:, None]
    if pos.shape[1] < 2:
        pos = np.column_stack([pos[:, 0], np.zeros(n)])
    pos = pos[:, :2]

    pos = pos - np.mean(pos, axis=0, keepdims=True)
    radius = np.max(np.linalg.norm(pos, axis=1))
    if np.isfinite(radius) and radius > 0:
        pos = pos / radius
    return pos


Extract weighted attention graphs and hidden-state projections for selected layers.


In [ ]:

def _extract_attention_graph_for_layer(
    outputs_attn,
    input_ids,
    layer_index,
    bos_token_id=None,
    drop_bos=True,
    renormalize_after_drop=True,
    head_weighting='mass',
):
    if outputs_attn.attentions is None:
        raise ValueError('Model did not return attentions. Use eager attention implementation.')

    num_attn_layers = len(outputs_attn.attentions)
    layer_use = int(layer_index)
    if layer_use < 0:
        layer_use += num_attn_layers
    if layer_use < 0 or layer_use >= num_attn_layers:
        raise ValueError(f'Invalid attention layer_index={layer_index} for {num_attn_layers} layers')

    layer_attn = outputs_attn.attentions[layer_use][0].float()
    token_start = get_token_start_index(input_ids, bos_token_id=bos_token_id, drop_bos=drop_bos)

    A = layer_attn[:, token_start:, token_start:]
    if renormalize_after_drop and token_start > 0:
        row_sums = A.sum(dim=-1, keepdim=True)
        row_sums = torch.where(row_sums == 0, torch.ones_like(row_sums), row_sums)
        A = A / row_sums

    W_heads = 0.5 * (A + A.transpose(-1, -2))
    if head_weighting == 'uniform':
        alpha = torch.full((A.shape[0],), 1.0 / A.shape[0], device=A.device, dtype=A.dtype)
    elif head_weighting == 'mass':
        s = A.sum(dim=(1, 2))
        s_sum = s.sum()
        if s_sum <= 0:
            alpha = torch.full((A.shape[0],), 1.0 / A.shape[0], device=A.device, dtype=A.dtype)
        else:
            alpha = s / s_sum
    else:
        raise ValueError("head_weighting must be 'mass' or 'uniform'")

    Wbar = (alpha[:, None, None] * W_heads).sum(dim=0)
    return Wbar.detach().cpu().numpy(), int(token_start), int(layer_use)


def _projection_result_from_hidden_states_for_layer(
    hidden_states,
    input_ids,
    layer_index,
    bos_token_id=None,
    drop_bos=True,
):
    signal_matrix, seq_len, used_layer = extract_hidden_signal_matrix(
        hidden_states,
        input_ids=input_ids,
        bos_token_id=bos_token_id,
        layer_index=layer_index,
        drop_bos=drop_bos,
    )
    svd_result = svd_projection_uncentered(signal_matrix, k=3)
    coords = np.asarray(svd_result['coords'], dtype=np.float32)[:seq_len]
    return {
        'seq_len': int(coords.shape[0]),
        'layer_index': int(used_layer),
        'coords': coords,
        'cumulative_energy_ratio': np.asarray(svd_result['cumulative_energy_ratio'], dtype=np.float32),
    }


def _make_across_layer_cache_key(data_name, frac, ctx_len, layer_indices, init_mode=None, spring_iterations=None):
    spring_iterations_use = _resolve_spring_iterations(spring_iterations)
    return (
        str(data_name),
        float(frac),
        int(ctx_len),
        tuple(int(layer_index) for layer_index in layer_indices),
        str(MODEL_NAME),
        bool(rgb_graph_drop_bos),
        bool(rgb_graph_renormalize_after_drop),
        str(rgb_graph_head_weighting),
        str(rgb_graph_layout_method),
        int(rgb_graph_layout_seed),
        str(rgb_graph_time_init_mode if init_mode is None else init_mode),
        int(spring_iterations_use),
        float(rgb_graph_layout_edge_threshold_factor),
    )


def compute_processed_across_layer_data(
    data_name,
    frac=CONTEXT_FRAC,
    layer_specs=LAYERS_TO_SAVE,
    layout_method=None,
    init_mode=None,
    spring_iterations=None,
):
    if data_name not in input_data_types:
        raise KeyError(f'Unknown data_name={data_name!r}. Available: {list(input_data_types)}')

    sequence = input_data_types[data_name]
    input_sequence, ctx_len = _build_input_sequence_from_frac(sequence, frac, N)

    attn_model = _get_attention_capable_model_for_graph_vis()
    attn_device = next(attn_model.parameters()).device
    num_model_layers = int(attn_model.config.num_hidden_layers)
    hidden_size_use = int(attn_model.config.hidden_size)
    layer_indices = resolve_layer_indices_to_save(num_model_layers, layer_specs=layer_specs)
    layer_indices_zero_based = layer_indices - 1
    cache_key = _make_across_layer_cache_key(
        data_name,
        frac,
        ctx_len,
        layer_indices=layer_indices,
        init_mode=init_mode,
        spring_iterations=spring_iterations,
    )
    if cache_key in across_layers_graph_internal_cache:
        return across_layers_graph_internal_cache[cache_key]

    inputs = tokenizer(input_sequence, return_tensors='pt').to(attn_device)
    input_ids = inputs.input_ids[0]
    input_ids_np = input_ids.detach().cpu().numpy()
    token_start = get_token_start_index(
        input_ids_np,
        bos_token_id=tokenizer.bos_token_id,
        drop_bos=rgb_graph_drop_bos,
    )
    token_ids = np.asarray(input_ids_np[token_start:], dtype=np.int64)
    is_delimiter = token_ids == comma_token_id

    layout_method_use = rgb_graph_layout_method if layout_method is None else str(layout_method)
    init_mode_use = rgb_graph_time_init_mode if init_mode is None else str(init_mode)
    spring_iterations_use = _resolve_spring_iterations(spring_iterations)
    pin_endpoints_use = bool(rgb_graph_time_init_pin_endpoints)
    node_count = int(token_ids.shape[0])
    num_svd_modes = min(node_count, hidden_size_use)

    checkpoint = load_or_initialize_graph_generation_checkpoint(
        data_name=data_name,
        frac=frac,
        layer_indices=layer_indices,
        node_count=node_count,
        num_svd_modes=num_svd_modes,
        is_delimiter=is_delimiter,
        layout_method=layout_method_use,
        init_mode=init_mode_use,
        spring_iterations=spring_iterations_use,
        pin_endpoints=pin_endpoints_use,
    )

    if np.all(checkpoint['completed_layers']):
        processed_data = processed_data_from_graph_checkpoint(checkpoint)
        across_layers_graph_internal_cache[cache_key] = processed_data
        del inputs, input_ids
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return processed_data

    with torch.no_grad():
        outputs = attn_model(
            **inputs,
            output_hidden_states=True,
            output_attentions=True,
            return_dict=True,
        )

    if outputs.attentions is None:
        raise ValueError('Model did not return attentions. Use eager attention implementation.')
    if outputs.hidden_states is None:
        raise ValueError('Model did not return hidden states.')

    num_attn_layers = len(outputs.attentions)
    if num_attn_layers != num_model_layers:
        raise ValueError(f'Model config reports {num_model_layers} layers, but returned {num_attn_layers} attention tensors.')

    completed_since_checkpoint = 0
    checkpoint_every = max(1, int(CHECKPOINT_EVERY_N_LAYERS))
    layer_iter = tqdm(
        list(enumerate(layer_indices_zero_based)),
        desc=f'{data_name} saved layers',
        unit='layer',
        leave=False,
        dynamic_ncols=True,
    )
    for saved_layer_offset, layer_idx in layer_iter:
        if checkpoint['completed_layers'][saved_layer_offset]:
            continue

        proj_result = _projection_result_from_hidden_states_for_layer(
            outputs.hidden_states,
            input_ids=input_ids,
            bos_token_id=tokenizer.bos_token_id,
            layer_index=int(layer_idx),
            drop_bos=rgb_graph_drop_bos,
        )
        Wbar, token_start_graph, used_attn_layer = _extract_attention_graph_for_layer(
            outputs,
            input_ids=input_ids,
            bos_token_id=tokenizer.bos_token_id,
            layer_index=int(layer_idx),
            drop_bos=rgb_graph_drop_bos,
            renormalize_after_drop=rgb_graph_renormalize_after_drop,
            head_weighting=rgb_graph_head_weighting,
        )

        if int(used_attn_layer) != int(layer_idx):
            raise ValueError(f'Layer mismatch: requested {layer_idx}, got attention layer {used_attn_layer}.')
        if int(proj_result['layer_index']) != int(layer_idx):
            raise ValueError(f'Hidden-state mismatch: requested {layer_idx}, got hidden-state {proj_result["layer_index"]}.')
        if int(token_start_graph) != int(token_start):
            raise ValueError(f'BOS token-start mismatch: graph={token_start_graph}, signal={token_start}.')
        if Wbar.shape[0] != node_count or proj_result['seq_len'] != node_count:
            raise ValueError(
                f'Node-count mismatch on layer {int(layer_idx) + 1}: '
                f'Wbar={Wbar.shape[0]}, signal={proj_result["seq_len"]}, tokens={node_count}.'
            )

        cumulative_energy = np.asarray(proj_result['cumulative_energy_ratio'], dtype=np.float32)
        if cumulative_energy.shape[0] != num_svd_modes:
            raise ValueError(f'SVD mode-count mismatch: got {cumulative_energy.shape[0]}, expected {num_svd_modes}.')

        checkpoint['Wbar'][saved_layer_offset] = np.asarray(Wbar, dtype=np.float32)
        checkpoint['svd_projection_coords'][saved_layer_offset] = np.asarray(proj_result['coords'], dtype=np.float32)
        checkpoint['layout_positions'][saved_layer_offset] = _build_topological_layout_from_attention_time_init(
            Wbar,
            init_mode=init_mode_use,
            method=layout_method_use,
            seed=rgb_graph_layout_seed,
            threshold_factor=rgb_graph_layout_edge_threshold_factor,
            spring_iterations=spring_iterations_use,
            pin_endpoints=pin_endpoints_use,
        ).astype(np.float32)
        checkpoint['svd_cumulative_energy_ratio'][saved_layer_offset] = cumulative_energy
        checkpoint['completed_layers'][saved_layer_offset] = True
        completed_since_checkpoint += 1

        if completed_since_checkpoint >= checkpoint_every:
            save_graph_generation_checkpoint(data_name, frac, checkpoint)
            completed_since_checkpoint = 0

    if completed_since_checkpoint > 0 or not graph_checkpoint_path(data_name, frac).exists():
        save_graph_generation_checkpoint(data_name, frac, checkpoint)

    if not np.all(checkpoint['completed_layers']):
        missing_layers = checkpoint['layer_indices'][~checkpoint['completed_layers']]
        raise RuntimeError(f'Incomplete checkpoint after generation; missing one-based layers: {missing_layers.tolist()}')

    processed_data = processed_data_from_graph_checkpoint(checkpoint)
    across_layers_graph_internal_cache[cache_key] = processed_data

    del outputs, inputs, input_ids
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return processed_data


Save, load, validate, and checkpoint the processed graph archives.


In [ ]:

def _atomic_savez_compressed(path, **arrays):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_name(path.name + '.tmp')
    with tmp_path.open('wb') as handle:
        np.savez_compressed(handle, **arrays)
    tmp_path.replace(path)


def save_processed_graph_data(data_name, frac, processed_data, data_dir=PROCESSED_DATA_DIR):
    path = graph_data_path(data_name, frac=frac, data_dir=data_dir)

    missing = [key for key in GRAPH_DATA_ARRAY_KEYS if key not in processed_data]
    if missing:
        raise KeyError(f'Missing required processed arrays: {missing}')

    arrays_to_save = {
        'layer_indices': np.asarray(processed_data['layer_indices'], dtype=np.int32),
        'Wbar': np.asarray(processed_data['Wbar'], dtype=np.float32),
        'layout_positions': np.asarray(processed_data['layout_positions'], dtype=np.float32),
        'svd_projection_coords': np.asarray(processed_data['svd_projection_coords'], dtype=np.float32),
        'svd_cumulative_energy_ratio': np.asarray(processed_data['svd_cumulative_energy_ratio'], dtype=np.float32),
        'is_delimiter': np.asarray(processed_data['is_delimiter'], dtype=bool),
    }
    _atomic_savez_compressed(path, **arrays_to_save)
    return path


def load_processed_graph_data(data_name, frac=CONTEXT_FRAC, data_dir=PROCESSED_DATA_DIR):
    path = graph_data_path(data_name, frac=frac, data_dir=data_dir)
    if not path.exists():
        raise FileNotFoundError(path)

    with np.load(path, allow_pickle=False) as npz:
        processed_data = {key: npz[key] for key in npz.files}

    missing = [key for key in GRAPH_DATA_ARRAY_KEYS if key not in processed_data]
    if missing:
        raise KeyError(f'{path} is missing required arrays: {missing}')
    extra = sorted(set(processed_data) - set(GRAPH_DATA_ARRAY_KEYS))
    if extra:
        raise KeyError(f'{path} contains unexpected arrays: {extra}')
    return processed_data


def processed_graph_data_complete(data_name, frac=CONTEXT_FRAC, data_dir=PROCESSED_DATA_DIR):
    try:
        processed_data = load_processed_graph_data(data_name, frac=frac, data_dir=data_dir)
    except (FileNotFoundError, OSError, KeyError, ValueError):
        return False

    layer_indices = np.asarray(processed_data['layer_indices'], dtype=np.int32)
    if layer_indices.size == 0:
        return False
    n_layers = int(layer_indices.size)
    n_nodes = int(np.asarray(processed_data['is_delimiter']).shape[0])
    n_modes = min(n_nodes, int(hidden_size)) if hidden_size is not None else np.asarray(processed_data['svd_cumulative_energy_ratio']).shape[1]
    expected_shapes = {
        'Wbar': (n_layers, n_nodes, n_nodes),
        'layout_positions': (n_layers, n_nodes, 2),
        'svd_projection_coords': (n_layers, n_nodes, 3),
        'svd_cumulative_energy_ratio': (n_layers, n_modes),
    }
    if num_layers is None:
        expected_layers = np.asarray(LAYERS_TO_SAVE, dtype=np.int32)
    else:
        expected_layers = resolve_layer_indices_to_save(int(num_layers), LAYERS_TO_SAVE)
    layers_match = np.array_equal(layer_indices, expected_layers)
    return layers_match and all(np.asarray(processed_data[key]).shape == shape for key, shape in expected_shapes.items())


def graph_checkpoint_path(data_name, frac=CONTEXT_FRAC, data_dir=PROCESSED_DATA_DIR):
    return Path(data_dir) / '_checkpoints' / f'{safe_data_stem(data_name, frac)}_graph_data.checkpoint.npz'


def initialize_graph_generation_checkpoint(layer_indices, node_count, num_svd_modes, is_delimiter):
    layer_indices = np.asarray(layer_indices, dtype=np.int32)
    n_layers = int(layer_indices.size)
    node_count = int(node_count)
    num_svd_modes = int(num_svd_modes)
    return {
        'layer_indices': layer_indices,
        'Wbar': np.zeros((n_layers, node_count, node_count), dtype=np.float32),
        'layout_positions': np.zeros((n_layers, node_count, 2), dtype=np.float32),
        'svd_projection_coords': np.zeros((n_layers, node_count, 3), dtype=np.float32),
        'svd_cumulative_energy_ratio': np.full((n_layers, num_svd_modes), np.nan, dtype=np.float32),
        'is_delimiter': np.asarray(is_delimiter, dtype=bool),
        'completed_layers': np.zeros((n_layers,), dtype=bool),
    }


def _npz_scalar_to_str(data, key):
    return str(np.asarray(data[key]).item())


def _checkpoint_metadata_matches(
    data,
    data_name,
    frac,
    layer_indices,
    node_count,
    num_svd_modes,
    is_delimiter,
    layout_method,
    init_mode,
    spring_iterations,
    pin_endpoints,
):
    layer_indices = np.asarray(layer_indices, dtype=np.int32)
    n_layers = int(layer_indices.size)
    node_count = int(node_count)
    num_svd_modes = int(num_svd_modes)

    if _npz_scalar_to_str(data, 'checkpoint_schema_version') != GRAPH_CHECKPOINT_SCHEMA_VERSION:
        return False
    if _npz_scalar_to_str(data, 'model_name') != MODEL_NAME:
        return False
    if _npz_scalar_to_str(data, 'data_name') != str(data_name):
        return False
    if not np.isclose(float(np.asarray(data['context_frac']).item()), float(frac)):
        return False
    if int(np.asarray(data['node_count']).item()) != node_count:
        return False
    if int(np.asarray(data['num_svd_modes']).item()) != num_svd_modes:
        return False
    if _npz_scalar_to_str(data, 'layout_method') != str(layout_method):
        return False
    if int(np.asarray(data['layout_seed']).item()) != int(rgb_graph_layout_seed):
        return False
    if not np.isclose(float(np.asarray(data['layout_edge_threshold_factor']).item()), float(rgb_graph_layout_edge_threshold_factor)):
        return False
    if _npz_scalar_to_str(data, 'init_mode') != str(init_mode):
        return False
    if int(np.asarray(data['spring_iterations']).item()) != int(spring_iterations):
        return False
    if bool(np.asarray(data['pin_endpoints']).item()) != bool(pin_endpoints):
        return False
    if bool(np.asarray(data['drop_bos']).item()) != bool(rgb_graph_drop_bos):
        return False
    if bool(np.asarray(data['renormalize_after_drop']).item()) != bool(rgb_graph_renormalize_after_drop):
        return False
    if _npz_scalar_to_str(data, 'head_weighting') != str(rgb_graph_head_weighting):
        return False
    if not np.array_equal(data['layer_indices'].astype(np.int32), layer_indices):
        return False
    if not np.array_equal(data['is_delimiter'].astype(bool), np.asarray(is_delimiter, dtype=bool)):
        return False

    expected_shapes = {
        'Wbar': (n_layers, node_count, node_count),
        'layout_positions': (n_layers, node_count, 2),
        'svd_projection_coords': (n_layers, node_count, 3),
        'svd_cumulative_energy_ratio': (n_layers, num_svd_modes),
        'completed_layers': (n_layers,),
    }
    return all(data[key].shape == shape for key, shape in expected_shapes.items())


def load_or_initialize_graph_generation_checkpoint(
    data_name,
    frac,
    layer_indices,
    node_count,
    num_svd_modes,
    is_delimiter,
    layout_method,
    init_mode,
    spring_iterations,
    pin_endpoints,
):
    path = graph_checkpoint_path(data_name, frac=frac)
    if path.exists():
        try:
            with np.load(path, allow_pickle=False) as data:
                if _checkpoint_metadata_matches(
                    data,
                    data_name=data_name,
                    frac=frac,
                    layer_indices=layer_indices,
                    node_count=node_count,
                    num_svd_modes=num_svd_modes,
                    is_delimiter=is_delimiter,
                    layout_method=layout_method,
                    init_mode=init_mode,
                    spring_iterations=spring_iterations,
                    pin_endpoints=pin_endpoints,
                ):
                    checkpoint = {
                        'layer_indices': data['layer_indices'].astype(np.int32),
                        'Wbar': data['Wbar'].astype(np.float32),
                        'layout_positions': data['layout_positions'].astype(np.float32),
                        'svd_projection_coords': data['svd_projection_coords'].astype(np.float32),
                        'svd_cumulative_energy_ratio': data['svd_cumulative_energy_ratio'].astype(np.float32),
                        'is_delimiter': data['is_delimiter'].astype(bool),
                        'completed_layers': data['completed_layers'].astype(bool),
                    }
                    done = int(checkpoint['completed_layers'].sum())
                    print(f'Resuming {data_name}: {done}/{int(checkpoint["layer_indices"].size)} saved layers already checkpointed.')
                    return checkpoint
                print(f'Ignoring incompatible checkpoint: {path}')
        except (OSError, KeyError, ValueError) as exc:
            print(f'Ignoring unreadable checkpoint {path}: {exc}')

    return initialize_graph_generation_checkpoint(
        layer_indices=layer_indices,
        node_count=node_count,
        num_svd_modes=num_svd_modes,
        is_delimiter=is_delimiter,
    )


def save_graph_generation_checkpoint(data_name, frac, checkpoint, data_dir=PROCESSED_DATA_DIR):
    path = graph_checkpoint_path(data_name, frac=frac, data_dir=data_dir)
    arrays_to_save = {
        'checkpoint_schema_version': np.asarray(GRAPH_CHECKPOINT_SCHEMA_VERSION),
        'model_name': np.asarray(MODEL_NAME),
        'data_name': np.asarray(str(data_name)),
        'context_frac': np.asarray(float(frac), dtype=np.float64),
        'node_count': np.asarray(checkpoint['is_delimiter'].size, dtype=np.int64),
        'num_svd_modes': np.asarray(checkpoint['svd_cumulative_energy_ratio'].shape[1], dtype=np.int64),
        'layout_method': np.asarray(str(rgb_graph_layout_method)),
        'layout_seed': np.asarray(rgb_graph_layout_seed, dtype=np.int64),
        'layout_edge_threshold_factor': np.asarray(rgb_graph_layout_edge_threshold_factor, dtype=np.float64),
        'init_mode': np.asarray(str(rgb_graph_time_init_mode)),
        'spring_iterations': np.asarray(rgb_graph_time_init_spring_iterations, dtype=np.int64),
        'pin_endpoints': np.asarray(bool(rgb_graph_time_init_pin_endpoints)),
        'drop_bos': np.asarray(bool(rgb_graph_drop_bos)),
        'renormalize_after_drop': np.asarray(bool(rgb_graph_renormalize_after_drop)),
        'head_weighting': np.asarray(str(rgb_graph_head_weighting)),
        'layer_indices': np.asarray(checkpoint['layer_indices'], dtype=np.int32),
        'Wbar': np.asarray(checkpoint['Wbar'], dtype=np.float32),
        'layout_positions': np.asarray(checkpoint['layout_positions'], dtype=np.float32),
        'svd_projection_coords': np.asarray(checkpoint['svd_projection_coords'], dtype=np.float32),
        'svd_cumulative_energy_ratio': np.asarray(checkpoint['svd_cumulative_energy_ratio'], dtype=np.float32),
        'is_delimiter': np.asarray(checkpoint['is_delimiter'], dtype=bool),
        'completed_layers': np.asarray(checkpoint['completed_layers'], dtype=bool),
    }
    _atomic_savez_compressed(path, **arrays_to_save)
    return path


def processed_data_from_graph_checkpoint(checkpoint):
    return {
        'layer_indices': np.asarray(checkpoint['layer_indices'], dtype=np.int32),
        'Wbar': np.asarray(checkpoint['Wbar'], dtype=np.float32),
        'layout_positions': np.asarray(checkpoint['layout_positions'], dtype=np.float32),
        'svd_projection_coords': np.asarray(checkpoint['svd_projection_coords'], dtype=np.float32),
        'svd_cumulative_energy_ratio': np.asarray(checkpoint['svd_cumulative_energy_ratio'], dtype=np.float32),
        'is_delimiter': np.asarray(checkpoint['is_delimiter'], dtype=bool),
    }


def delete_graph_generation_checkpoint(data_name, frac=CONTEXT_FRAC, data_dir=PROCESSED_DATA_DIR):
    path = graph_checkpoint_path(data_name, frac=frac, data_dir=data_dir)
    if path.exists():
        path.unlink()


Generate or reuse every required input archive at $N = 500$.


In [ ]:

generated_paths = []
skipped_paths = []

for data_name in tqdm(DATA_ORDER, desc='Selected-layer graph/SVD data', unit='input'):
    output_path = graph_data_path(data_name, frac=CONTEXT_FRAC)
    if processed_graph_data_complete(data_name, frac=CONTEXT_FRAC):
        skipped_paths.append(output_path)
        delete_graph_generation_checkpoint(data_name, frac=CONTEXT_FRAC)
        continue

    if output_path.exists():
        print(f'Rebuilding incomplete or outdated archive: {output_path}')

    processed = compute_processed_across_layer_data(
        data_name,
        frac=CONTEXT_FRAC,
        layer_specs=LAYERS_TO_SAVE,
        init_mode=rgb_graph_time_init_mode,
        spring_iterations=rgb_graph_time_init_spring_iterations,
    )
    saved_path = save_processed_graph_data(data_name, CONTEXT_FRAC, processed)
    delete_graph_generation_checkpoint(data_name, frac=CONTEXT_FRAC)
    generated_paths.append(saved_path)

print(f'Generated {len(generated_paths)} archive(s); skipped {len(skipped_paths)} existing archive(s).')
print(f'Processed graph data directory: {PROCESSED_DATA_DIR}')
